In [1]:
# Install the PDF reader and the Google GenAI SDK
!pip install PyMuPDF pillow google-genai
!pip install sentence-transformers
!pip install PyMuPDF google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 83.6 MB/s eta 0:00:00


# **cv and job parsing**

In [2]:
import os
import json
import fitz
from google import genai
from google.genai import types
from google.colab import userdata

try:
    # Load the key securely from Colab Secrets
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    client = genai.Client()
    MODEL_NAME = "gemini-2.5-flash"
    print("Gemini Client initialized successfully using Colab Secrets.")
except Exception as e:
    print(f"Error initializing Gemini Client: {e}")
    print("Please ensure you have set your 'GEMINI_API_KEY' in Colab Secrets.")

Gemini Client initialized successfully using Colab Secrets.


In [3]:
import json
from google.genai import types

# --- JD TECH SKILLS ONLY SCHEMA ---
JD_TECH_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "jd_tech_skills": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            description=(
                "A deduplicated list of ONLY technical skills found in the job description. "
                "Include programming languages, frameworks, libraries, databases, cloud platforms, DevOps tools, "
                "data tools, ML/AI tools, APIs, OS/tools, etc. Exclude soft skills."
            ),
        ),
    },
    required=["jd_tech_skills"],
)

def parse_jd_tech_skills_with_llm(jd_path: str):
    # 1) Extract JD text
    text = ""
    try:
        if jd_path.lower().endswith(".pdf"):
            import fitz  # PyMuPDF
            doc = fitz.open(jd_path)
            for page in doc:
                text += page.get_text()
        else:
            with open(jd_path, "r", encoding="utf-8") as f:
                text = f.read()

        # Safety truncate
        if len(text) > 15000:
            text = text[:15000]

        if not text.strip():
            return {"error": "No text found in JD."}
    except Exception as e:
        return {"error": str(e)}

    # 2) Prompt: TECH ONLY
    prompt = f"""
You are a senior technical recruiter and ATS parser.

Extract ONLY technical skills from the job description text and return JSON with exactly:
{{"jd_tech_skills": ["..."]}}

IMPORTANT: Be EXHAUSTIVE. Include both:
1) Concrete technologies/tools (languages, frameworks, libraries, databases, cloud services, DevOps tools, platforms).
2) Technical concepts/domains explicitly mentioned (e.g., API design/APIs, cloud-native services, multitenancy, distributed systems, infrastructure security, cost optimization, observability, automation, data warehouses, metrics analysis).

Rules:
- Use the ENTIRE text (company intro + responsibilities + qualifications). Do not limit to a single section.
- Technical items ONLY. EXCLUDE soft skills/behavioral traits (communication, teamwork, leadership, motivation, professionalism, etc.).
- EXCLUDE benefits/perks, culture statements, and general HR text (salary, insurance, gym, celebrations, equal opportunity, etc.).
- Keep items short and “skill-like”:
  - Prefer noun phrases of 1–6 words (e.g., "API design", "cloud cost optimization", "infrastructure security").
  - Avoid full sentences.
- Deduplicate aggressively: include each skill ONCE.
- Normalize common names:
  - "Amazon Web Services" -> "AWS"
  - "Google Cloud Platform" / "Google Cloud" -> "GCP"
  - "Postgres" -> "PostgreSQL"
  - "Infrastructure as Code" -> "IaC"
  - "Shell scripting" -> "Shell"
- Keep acronyms when common (CI/CD, ECS).
- If a line lists examples like "Shell, Bash, Python", include each separately.

Return ONLY valid JSON matching the schema.

JOB DESCRIPTION TEXT:
---
{text}
"""

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=JD_TECH_SCHEMA,
            ),
        )
        data = json.loads(response.text)

        # Final safety: ensure unique + clean strings
        skills = data.get("jd_tech_skills", [])
        cleaned = []
        seen = set()
        for s in skills:
            if not isinstance(s, str):
                continue
            s2 = s.strip()
            if not s2:
                continue
            key = s2.lower()
            if key not in seen:
                seen.add(key)
                cleaned.append(s2)

        return {"jd_tech_skills": cleaned}

    except Exception as e:
        return {"error": f"API Error: {e}"}


## **Interview** **Preparation**

In [4]:
# # =========================
# # 0) CONFIG: put your paths
# # =========================
# JD_PATH = "/content/DevOps.txt"
# # =========================
# # 1) Parse JD TECH SKILLS ONLY with Gemini
# # =========================
# print("Parsing Job Description (TECH skills only)...")
# jd_data = parse_jd_tech_skills_with_llm(JD_PATH)

# if "error" in jd_data:
#     raise ValueError(f"JD parse error: {jd_data['error']}")

# # =========================
# # 2) (Optional) quick preview
# # =========================
# jd_tech_skills = jd_data.get("jd_tech_skills", [])
# print(f"\nExtracted JD technical skills: {len(jd_tech_skills)}")
# print(" - " + "\n - ".join(jd_tech_skills[:30]) + ("" if len(jd_tech_skills) <= 30 else "\n - ..."))

In [5]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import defaultdict
import re
from sentence_transformers import SentenceTransformer

# Load a lightweight, high-quality model for text similarity
model_sim = SentenceTransformer('all-MiniLM-L6-v2')
# --------------------
# Helpers
# --------------------
def is_skill_like(text: str, max_tokens=6, max_chars=45):
    """Light filter to remove sentence fragments; keep skill/tool-like strings."""
    if not text:
        return False
    t = str(text).strip().lower()
    t = " ".join(t.split())

    if len(t) > max_chars:
        return False
    if len(t.split()) > max_tokens:
        return False
    if len(t) < 2:
        return False

    # filter sentence-like fragments
    if re.search(r"\b(i|we|they|built|developed|implemented|responsible|manage|leading)\b", t):
        return False

    return True

def normalize_skill(x: str) -> str:
    x = str(x).strip().lower()
    x = " ".join(x.split())
    return x

def clean_list(xs):
    """Dedup while keeping order (assumes normalized)."""
    out, seen = [], set()
    for x in xs:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def build_skill_candidates_from_jd(jd_data):
    """
    JD-only TECH skill list:
    - read from jd_data["jd_tech_skills"]
    - normalized + filtered + deduped
    """
    raw = jd_data.get("jd_tech_skills", []) or []

    cleaned = []
    for x in raw:
        nx = normalize_skill(x)
        if is_skill_like(nx):
            cleaned.append(nx)

    return clean_list(cleaned)

def pick_k_silhouette(embeddings, k_min=2, k_max=10, random_state=42):
    n = embeddings.shape[0]
    if n < 3:
        return 1

    k_max = min(k_max, n - 1)
    if k_max < 2:
        return 1

    k_min = max(2, min(k_min, k_max))

    best_k, best_score = k_min, -1
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init="auto", random_state=random_state)
        labels = km.fit_predict(embeddings)
        if len(set(labels)) < 2:
            continue
        score = silhouette_score(embeddings, labels, metric="cosine")
        if score > best_score:
            best_score, best_k = score, k
    return best_k

def cluster_skills_kmeans(skills, model_sim, k=None, k_min=2, k_max=10, random_state=42):
    skills = clean_list(skills)

    if len(skills) == 0:
        return {"k": 0, "clusters": {}}
    if len(skills) == 1:
        return {"k": 1, "clusters": {0: skills}}

    emb = model_sim.encode(skills, convert_to_numpy=True, normalize_embeddings=True)

    if k is None:
        k = pick_k_silhouette(emb, k_min=k_min, k_max=k_max, random_state=random_state)
        if k == 1:
            return {"k": 1, "clusters": {0: skills}}

    km = KMeans(n_clusters=k, n_init="auto", random_state=random_state)
    labels = km.fit_predict(emb)

    clusters = defaultdict(list)
    for skill, lab in zip(skills, labels):
        clusters[int(lab)].append(skill)

    return {"k": k, "clusters": dict(clusters)}

# --------------------
# Run clustering (JD TECH ONLY)
# --------------------
# skills_for_clustering = build_skill_candidates_from_jd(jd_data)

# print("JD TECH skills to cluster:", len(skills_for_clustering))
# for s in skills_for_clustering:
#     print("-", s)

# clustering_result = cluster_skills_kmeans(skills_for_clustering, model_sim, k=None, k_min=2, k_max=10)
# print("\nChosen K:", clustering_result["k"])
# for cid, items in sorted(clustering_result["clusters"].items(), key=lambda x: len(x[1]), reverse=True):
#     print(f"\nCluster {cid} (size={len(items)}):")
#     for s in items:
#         print(" -", s)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# RAG T5


In [6]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.0 MB/s eta 0:00:00


In [7]:
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

groq_client  = Groq(api_key=GROQ_API_KEY)

In [8]:
def get_completion(prompt, model="llama-3.1-8b-instant"):
    """
    A simple helper to send prompts to Groq LLaMA 3.1 models.
    Default: llama-3.1-8b-instant (fast & free)
    Other options:
        - llama-3.1-70b-versatile
        - llama-3.1-405b-reasoning (strong thinking)
    """
    chat_completion = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return chat_completion.choices[0].message.content

In [9]:
# ── STEP 1: Install & Import ──────────────────────────────
!pip install sentence-transformers faiss-cpu -q

import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.8 MB/s eta 0:00:00


In [10]:
import json
import pandas as pd
import re
from collections import defaultdict, Counter

def load_all_datasets():
    unified = []

    # ── Dataset 1: Original JSON ────────────────────────────
    print("📂 Loading Dataset 1 (Main JSON)...")
    with open("/content/Cleaned_Computer_Science_Questions.json", "r") as f:
        raw = json.load(f)
    data1 = raw if isinstance(raw, list) else next(iter(raw.values()))
    for r in data1:
        q = str(r.get("question", "")).strip()
        a = str(r.get("answer", "")).strip()
        if q and a:
            unified.append({
                "question": q,
                "answer":   a,
                "subject":  r.get("subject", r.get("subfield", "General")),
                "field":    "Computer Science",
                "source":   "dataset_main"
            })
    print(f"   ✅ {len(data1)} records")

    # ── Dataset 2: Cleaned ML CSV ───────────────────────────
    print("📂 Loading Dataset 2 (Machine Learning CSV)...")
    df2 = pd.read_csv("/content/dataset_cleaned.csv")
    for _, row in df2.iterrows():
        q = str(row.get("question", "")).strip()
        a = str(row.get("answer", "")).strip()
        if q and a:
            unified.append({
                "question": q,
                "answer":   a,
                "subject":  "Machine Learning",
                "field":    "Computer Science",
                "source":   "dataset_ml"
            })
    print(f"   ✅ {len(df2)} records")

    # ── Dataset 3: Cleaned JSONL ────────────────────────────
    print("📂 Loading Dataset 3 (General CS JSON)...")
    with open("/content/train_cleaned.json", "r") as f:
        data3 = json.load(f)
    for r in data3:
        q = str(r.get("question", "")).strip()
        a = str(r.get("answer", "")).strip()
        if q and a:
            unified.append({
                "question": q,
                "answer":   a,
                "subject":  "General Computer Science",
                "field":    "Computer Science",
                "source":   "dataset_jsonl"
            })
    print(f"   ✅ {len(data3)} records")

    # ── Dataset 4: Cleaned Software CSV ─────────────────────
    print("📂 Loading Dataset 4 (Software Engineering CSV)...")
    df4 = pd.read_csv("/content/software_questions_cleaned.csv")
    for _, row in df4.iterrows():
        q = str(row.get("question", "")).strip()
        a = str(row.get("answer", "")).strip()
        cat  = str(row.get("category",   "Software Engineering")).strip()
        diff = str(row.get("difficulty", "")).strip()
        if q and a:
            unified.append({
                "question":   q,
                "answer":     a,
                "subject":    cat if cat != "nan" else "Software Engineering",
                "field":      "Computer Science",
                "source":     "dataset_software",
                "difficulty": diff if diff != "nan" else ""
            })
    print(f"   ✅ {len(df4)} records")

    # ── Remove cross-dataset duplicates ─────────────────────
    print(f"\n🔍 Removing cross-dataset duplicates...")
    before = len(unified)
    seen = set()
    deduped = []
    for r in unified:
        key = re.sub(r'[^\w\s]', '', r["question"].lower().strip())
        key = " ".join(key.split())
        if key not in seen:
            seen.add(key)
            deduped.append(r)
    after = len(deduped)
    print(f"   Removed {before - after} cross-dataset duplicates")

    # ── Summary ──────────────────────────────────────────────
    print(f"\n{'='*40}")
    print(f"📦 TOTAL RECORDS: {after}")
    for src, count in Counter(r["source"] for r in deduped).items():
        print(f"   {src}: {count}")
    print(f"{'='*40}")

    return deduped


data = load_all_datasets()

📂 Loading Dataset 1 (Main JSON)...
   ✅ 1283 records
📂 Loading Dataset 2 (Machine Learning CSV)...
   ✅ 166 records
📂 Loading Dataset 3 (General CS JSON)...
   ✅ 1264 records
📂 Loading Dataset 4 (Software Engineering CSV)...
   ✅ 174 records

🔍 Removing cross-dataset duplicates...
   Removed 113 cross-dataset duplicates

📦 TOTAL RECORDS: 2774
   dataset_main: 1283
   dataset_ml: 166
   dataset_jsonl: 1263
   dataset_software: 62


In [11]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedder loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedder loaded!


In [12]:
print("🔄 Building FAISS index...")
corpus = []
for record in data:
    text = f"{record.get('subject','')} {record.get('field','')} {record.get('question','')}".strip()
    corpus.append(text)

corpus_embeddings = embedder.encode(
    corpus,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings)
print(f"✅ FAISS index ready with {index.ntotal} entries")

🔄 Building FAISS index...


Batches:   0%|          | 0/87 [00:00<?, ?it/s]

✅ FAISS index ready with 2774 entries


In [13]:
# ── STEP 4: RAG Retrieval Function ───────────────────────
def retrieve_relevant_questions(skills, index, data, embedder, top_k=3):
    """Find top_k most relevant Q&A pairs for given skills."""
    query = "interview question about " + ", ".join(skills)
    query_embedding = embedder.encode([query], normalize_embeddings=True)

    scores, indices = index.search(query_embedding, top_k)

    retrieved = []
    for idx in indices[0]:
        record = data[idx]
        retrieved.append({
            "question": record.get("question", ""),
            "answer":   record.get("answer", ""),
            "subject":  record.get("subject", ""),
        })
    return retrieved

In [14]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load flan-t5-large as generator
gen_model_name = "google/flan-t5-large"
gen_tokenizer  = T5Tokenizer.from_pretrained(gen_model_name)
gen_model      = T5ForConditionalGeneration.from_pretrained(gen_model_name).to(device)
print("✅ flan-t5-large loaded!")

Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ flan-t5-large loaded!


In [36]:
def generate_ideal_answer_groq(question, skills):
    skills_str = ", ".join(skills)

    prompt = (
        f"You are a senior software engineer conducting a technical interview.\n"
        f"The candidate is being evaluated on: {skills_str}\n\n"
        f"Interview Question: {question}\n\n"
        f"Provide a model answer that a strong candidate should give.\n"
        f"Requirements:\n"
        f"- Be technically accurate and specific\n"
        f"- Cover the key concepts related to {skills_str}\n"
        f"- Write 4-6 clear sentences\n"
        f"- Do not include phrases like 'Great question' or 'As an AI'\n"
        f"- Go straight to the answer\n\n"
        f"Answer:"
    )

    answer = get_completion(prompt, model="llama-3.3-70b-versatile")  # ← updated model
    return answer.strip()

def retrieve_relevant_questions(skills, index, data, embedder, top_k=3):
    """Find top_k most relevant Q&A pairs for given skills."""
    query = "interview question about " + ", ".join(skills)
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding, top_k)
    retrieved = []
    for idx in indices[0]:
        record = data[idx]
        retrieved.append({
            "question": record.get("question", ""),
            "answer":   record.get("answer", ""),
            "subject":  record.get("subject", ""),
        })
    return retrieved


def generate_rag_question(cluster_id, skills, index, data, embedder, difficulty="intermediate"):
    retrieved = retrieve_relevant_questions(skills, index, data, embedder, top_k=3)

    context = ""
    for i, r in enumerate(retrieved):
        clean_q = r['question'].replace("A:", "").replace("Q:", "").split("|")[0].strip()
        clean_a = r['answer'][:200].strip()
        context += f"Example {i+1}: Q: {clean_q} A: {clean_a}\n\n"

    skills_str = ", ".join(skills)

    # ── Step 1: Generate question with flan-t5 ───────────────
    question_prompt = (
        f"Generate a {difficulty} level technical interview question "
        f"about {skills_str}. "
        f"The question must be directly about {skills_str}. "
        f"Use these examples as style reference only: {context}"
        f"Generate one new question without any prefix like Q: or A::"
    )

    inputs = gen_tokenizer(
        question_prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    q_outputs = gen_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        min_length=20,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=2.5,
        length_penalty=1.0,
        early_stopping=True,
    )
    question = gen_tokenizer.decode(q_outputs[0], skip_special_tokens=True).strip()
    question = question.replace("A:", "").replace("Q:", "").split("|")[0].strip()

    # ── Step 2: Generate ideal answer with Groq LLaMA ────────
    print(f"     💭 Generating ideal answer with LLaMA...")
    ideal_answer = generate_ideal_answer_groq(question, skills)

    return {
        "cluster_id": cluster_id,
        "skills":     skills,
        "question":   question,
        "answer":     ideal_answer,
        "source":     [r["subject"] for r in retrieved]
    }


def question_is_relevant(question, skills):
    """Check if generated question is relevant to the skills."""
    question_lower = question.lower()
    skills_words = " ".join(skills).lower().split()
    return any(word in question_lower for word in skills_words)


def questions_are_similar(q1, q2, threshold=0.7):
    words1 = set(q1.lower().split())
    words2 = set(q2.lower().split())
    overlap = len(words1 & words2) / max(len(words1), len(words2))
    return overlap > threshold

def normalize_text_basic(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"[^\w\s/+\-\.]", " ", x)
    x = " ".join(x.split())
    return x


def exact_match_score(record, skill):
    """
    Exact/lexical matching using only fields available in our datasets:
    subject, field, question
    """
    skill_text = normalize_text_basic(skill)

    # Your dataset fields only
    record_subject  = normalize_text_basic(record.get("subject", ""))
    record_field    = normalize_text_basic(record.get("field", ""))
    record_question = normalize_text_basic(record.get("question", ""))

    score = 0.0

    # ── Subject field (strongest signal) ─────────────────────
    if skill_text == record_subject:
        score += 5.0
    elif skill_text and skill_text in record_subject:
        score += 3.0

    # ── Field (medium signal) ─────────────────────────────────
    if skill_text == record_field:
        score += 3.5
    elif skill_text and skill_text in record_field:
        score += 2.0

    # ── Question text (weak fallback signal) ──────────────────
    if skill_text and skill_text in record_question:
        score += 0.8

    return score


def is_valid_record(record):
    """
    Validates that a record has the minimum required fields
    to be used in question generation.
    """
    question = record.get("question", "").strip()
    answer   = record.get("answer", "").strip()
    return bool(question) and bool(answer)


def retrieve_exact_matches(skill, data, top_k=20):
    """
    Retrieves top_k records that best match the given skill
    using exact/lexical matching on subject, field, and question fields.
    """
    candidates = []

    for record in data:
        if not is_valid_record(record):
            continue

        score = exact_match_score(record, skill)
        if score > 0:
            candidates.append((score, record))

    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[:top_k]



def retrieve_exact_matches_for_cluster(skills, data, top_k=5):
    """
    Retrieves top_k records that best match ALL skills in a cluster
    by aggregating exact match scores across all skills.

    Input:  skills — list of skills from one cluster
    Output: list of top matching records
    """
    score_map = {}  # record index → total score

    for skill in skills:
        matches = retrieve_exact_matches(skill, data, top_k=20)
        for score, record in matches:
            # Use question as unique key
            key = record.get("question", "").strip().lower()
            if key not in score_map:
                score_map[key] = {"score": 0.0, "record": record}
            score_map[key]["score"] += score  # accumulate scores

    # Sort by total score descending
    sorted_matches = sorted(
        score_map.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    return [(item["score"], item["record"]) for item in sorted_matches[:top_k]]


def get_question_for_cluster(cluster_id, skills, index, data, embedder, difficulty="intermediate"):
    """
    Hybrid approach:
    - If FAISS finds a strong match (score >= 0.75) → use dataset question directly
    - Otherwise → generate with flan-t5 + LLaMA
    """
     # ── Step 1: Exact match across cluster skills ─────────────
    exact_matches = retrieve_exact_matches_for_cluster(skills, data, top_k=5)

    if exact_matches:
        best_score, best_record = exact_matches[0]
        if best_score >= 5.0:  # strong exact match found
            question = best_record.get("question", "").replace("A:", "").replace("Q:", "").split("|")[0].strip()
            answer   = best_record.get("answer", "").strip()
            if question and answer:
                print(f"  📖 Exact match found (score: {best_score:.1f}): {question[:60]}...")
                return {
                    "cluster_id":  cluster_id,
                    "skills":      skills,
                    "question":    question,
                    "answer":      answer,
                    "source":      "dataset_exact",
                    "match_score": best_score
                }


    skills_str = ", ".join(skills)
    query = f"interview question about {skills_str}"
    query_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_emb, 1)
    match_score = float(scores[0][0])
    best_match  = data[indices[0][0]]

    # ── Path 2: Strong match in dataset → use directly ───────
    if match_score >= 0.75:
        question = best_match.get("question", "").replace("A:", "").replace("Q:", "").split("|")[0].strip()
        answer   = best_match.get("answer", "").strip()

        # Only use if question and answer are not empty
        if question and answer:
            print(f"  📚 Using dataset question directly (match: {match_score:.2f})")
            return {
                "cluster_id": cluster_id,
                "skills":     skills,
                "question":   question,
                "answer":     answer,
                "source":     "dataset_direct",
                "match_score": match_score
            }

    # ── Path 3: No strong match → generate with flan-t5 + LLaMA
    print(f"  🔄 Generating new question (best match: {match_score:.2f})")
    return generate_rag_question(cluster_id, skills, index, data, embedder, difficulty)


def generate_rag_questions_for_clusters(clustering_result, index, data, embedder, questions_per_cluster=2):
    all_results  = []
    difficulties = ["junior", "intermediate", "senior"]

    for cid, skills in clustering_result["clusters"].items():
        print(f"\n📦 Cluster {cid} — skills: {skills}")

        actual_questions   = 1 if len(skills) <= 2 else questions_per_cluster
        generated_questions = []

        for i in range(actual_questions):
            difficulty = difficulties[i % len(difficulties)]

            for attempt in range(5):
                if attempt > 0 and len(skills) > 2:
                    import random
                    skills_subset = random.sample(skills, max(2, len(skills) // 2))
                else:
                    skills_subset = skills

                result = get_question_for_cluster(
                    cid, skills_subset, index, data, embedder, difficulty
                )

                is_duplicate = any(
                    questions_are_similar(result["question"], prev_q)
                    for prev_q in generated_questions
                )
                is_relevant = question_is_relevant(result["question"], skills_subset)

                if not is_duplicate and is_relevant:
                    generated_questions.append(result["question"])
                    all_results.append(result)
                    source_label = "📚 dataset" if result.get("source") == "dataset_direct" else "🤖 generated"
                    print(f"  ✅ Q{i+1} ({difficulty}) [{source_label}]: {result['question'][:100]}...")
                    break
                elif is_duplicate:
                    print(f"  ⚠️ Attempt {attempt+1} duplicate, retrying...")
                elif not is_relevant:
                    print(f"  ⚠️ Attempt {attempt+1} off-topic, retrying...")
            else:
                print(f"  ⏭️ Skipped Q{i+1} — couldn't generate relevant unique question")

    return all_results



# ── Run ───────────────────────────────────────────────────────
# all_questions = generate_rag_questions_for_clusters(
#     clustering_result, index, data, embedder,
#     questions_per_cluster=2
# )

# # ── Print Results ─────────────────────────────────────────────
# print("\n" + "="*60)
# print("📋 FINAL INTERVIEW QUESTIONS (RAG)")
# print("="*60)
# for r in all_questions:
#     print(f"\n🔹 Cluster {r['cluster_id']} | Skills: {', '.join(r['skills'])}")
#     print(f"   📚 Retrieved from: {r['source']}")
#     print(f"   Q: {r['question']}")
#     print(f"   A: {r['answer']}")
#     print("-"*60)
# # Run this and manually score each question
# for i, q in enumerate(all_questions):
#     print(f"\nQ{i+1}: {q['question']}")
#     print(f"Skills: {q['skills']}")
#     print(f"Source: {q.get('source', 'generated')}")

── Exact matches ──
  Score: 5.8 | Subject: Docker | Q: Explain the limitations and trade-offs of using Docker's content addressable sto
  Score: 5.8 | Subject: Docker | Q: What is the primary benefit of using Docker containers over traditional virtual 
  Score: 5.8 | Subject: Docker | Q: How would you optimize the resource utilization of a Docker container running a 

── Full pipeline ──
  📖 Exact match found (score: 5.8): Explain the limitations and trade-offs of using Docker's con...
  Source  : dataset_exact
  Question: Explain the limitations and trade-offs of using Docker's content addressable storage (CAS) mechanism to store and manage persistent data in a Docker container, and discuss a scenario where another storage approach might be more suitable.
  Answer  : Docker's content addressable storage (CAS) mechanism, also known as union file system, is a powerful and efficient way to manage and distribute container images. It allows for the creation of layers of read-only filesyst

# Interactive Interview Session

In [16]:
# # ── CELL 0: Interactive Interview Session ───────────────────

# def run_interview_session(all_questions):
#     """
#     Displays questions one by one from RAG output
#     and collects user answers.

#     all_questions: list of dicts from generate_rag_questions_for_clusters()
#     Each dict contains: cluster_id, skills, question, answer

#     Returns: interview_data ready for evaluate_interview()
#     """
#     interview_data = []
#     total = len(all_questions)

#     print("="*60)
#     print("🎤 INTERVIEW SESSION STARTED")
#     print("="*60)
#     print(f"You will be asked {total} questions.")
#     print("Type your answer and press Enter twice when done.\n")

#     for i, item in enumerate(all_questions):
#         print(f"\n{'─'*60}")
#         print(f"Question {i+1} of {total}")
#         print(f"📦 Topic: {', '.join(item['skills'])}")
#         print(f"{'─'*60}")
#         print(f"❓ {item['question']}")
#         print(f"{'─'*60}")
#         print("✏️  Your answer (press Enter twice when done):")

#         # Collect multi-line answer
#         lines = []
#         while True:
#             line = input()
#             if line == "":
#                 if lines:
#                     break
#             else:
#                 lines.append(line)

#         user_answer = " ".join(lines).strip()

#         # Handle empty answer
#         if not user_answer:
#             user_answer = "I don't know"
#             print("⚠️  No answer recorded — marked as 'I don't know'")
#         else:
#             print("✅ Answer recorded!")

#         # ← must be inside the for loop, indented with 8 spaces
#         interview_data.append({
#             "cluster_id":     item["cluster_id"],
#             "skills":         item["skills"],
#             "question":       item["question"],
#             "user_answer":    user_answer,
#             "correct_answer": item["answer"],
#         })

#     print(f"\n{'='*60}")
#     print(f"✅ Interview complete! {total} questions answered.")
#     print(f"{'='*60}")
#     print("🔍 Running evaluation...\n")

#     return interview_data


# # ── Run the interview session ────────────────────────────────
# interview_data = run_interview_session(all_questions)




# Evaluation and Feedback
 Hybrid Evaluation System — Interview Preparation Module

 Input:  question + user_answer + correct_answer (per cluster)

 Output: per-cluster skill feedback report



In [17]:
# ── CELL 1: Install Libraries ────────────────────────────────
!pip install bert-score sentence-transformers transformers torch -q




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


In [18]:
# ── CELL 2: Imports ──────────────────────────────────────────
import torch
import numpy as np
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from collections import defaultdict


In [19]:
import os
import torch
import numpy as np
from bert_score import BERTScorer
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

# ── Load Embedder ─────────────────────────────────────────────
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedder loaded")

# ── Load BERTScorer once ──────────────────────────────────────
print("Loading BERTScore model...")
bert_scorer = BERTScorer(
    model_type="distilbert-base-uncased",
    lang="en",
    rescale_with_baseline=False,
    device=device
)
print("✅ BERTScore model loaded")

# ── Load NLI model ────────────────────────────────────────────
nli_model = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-base",
    device=0 if device == "cuda" else -1
)
print("✅ NLI model loaded")


🖥️  Device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedder loaded
Loading BERTScore model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

✅ NLI model loaded


In [20]:
# ── CELL 4: Individual Metric Functions ─────────────────────

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from bert_score import BERTScorer

print("Loading BERTScore model once...")
bert_scorer = BERTScorer(
    model_type="bert-base-uncased",
    lang="en",
    rescale_with_baseline=True,
    device=device
)
print("✅ BERTScore model loaded")

def compute_bertscore(user_answer, correct_answer):
    import builtins
    scorer = builtins.bert_scorer_global
    P, R, F1 = scorer.score([user_answer], [correct_answer])
    return float(F1[0])

def compute_semantic_similarity(user_answer, correct_answer):
    import builtins
    emb = builtins.embedder_global
    emb_user    = emb.encode(user_answer,    convert_to_tensor=True)
    emb_correct = emb.encode(correct_answer, convert_to_tensor=True)
    similarity  = util.cos_sim(emb_user, emb_correct)
    return float(similarity[0][0])

def compute_nli_score(user_answer, correct_answer):
    import builtins
    nli = builtins.nli_model_global

    input_1 = f"{user_answer} [SEP] {correct_answer}"
    input_2 = f"{correct_answer} [SEP] {user_answer}"

    result_1 = nli(input_1, truncation=True, max_length=512)[0]
    result_2 = nli(input_2, truncation=True, max_length=512)[0]

    def label_to_score(result):
        label = result["label"].lower()
        score = result["score"]
        if "entail" in label:
            return score
        elif "neutral" in label:
            return score * 0.5
        else:
            return 1 - score

    return max(label_to_score(result_1), label_to_score(result_2))

print("✅ Metric functions updated to use builtins")

def compute_rouge_l(user_answer, correct_answer):
    """
    Computes ROUGE-L score (longest common subsequence).
    Returns float in [0, 1].
    """
    def lcs_length(a, b):
        a_tokens = a.lower().split()
        b_tokens = b.lower().split()
        m, n = len(a_tokens), len(b_tokens)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if a_tokens[i-1] == b_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        return dp[m][n]

    user_tokens    = user_answer.lower().split()
    correct_tokens = correct_answer.lower().split()

    if not user_tokens or not correct_tokens:
        return 0.0

    lcs    = lcs_length(user_answer, correct_answer)
    prec   = lcs / len(user_tokens)
    recall = lcs / len(correct_tokens)

    if prec + recall == 0:
        return 0.0

    return 2 * prec * recall / (prec + recall)


Loading BERTScore model once...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERTScore model loaded
✅ Metric functions updated to use builtins


In [21]:
# ── Replace Cell 5: Hybrid Score ─────────────────────────────
def compute_hybrid_score(user_answer, correct_answer, question):

    bertscore   = compute_bertscore(user_answer, correct_answer)
    semantic    = compute_semantic_similarity(user_answer, correct_answer)
    nli         = compute_nli_score(user_answer, correct_answer)
    rouge       = compute_rouge_l(user_answer, correct_answer)
    q_relevance = compute_semantic_similarity(user_answer, question)

    # Semantic similarity is the most reliable metric here
    # BERTScore and ROUGE are secondary signals only
    final_score = (
        0.50 * semantic    +   # most reliable — 0.818 for correct answers
        0.25 * q_relevance +   # checks answer is on topic
        0.15 * bertscore   +   # token overlap signal
        0.10 * nli             # logical check — reduced weight
    )

    # ROUGE removed entirely — too harsh on paraphrasing

    return {
        "hybrid_score": round(final_score,  3),
        "bertscore":    round(bertscore,    3),
        "semantic_sim": round(semantic,     3),
        "nli_score":    round(nli,          3),
        "rouge_l":      round(rouge,        3),
        "q_relevance":  round(q_relevance,  3),
    }

In [22]:
# ── CELL 6: Grade Function ───────────────────────────────────

def get_grade(score):
    if score >= 0.78:
        return "Excellent ✅"
    elif score >= 0.68:
        return "Good 👍"
    elif score >= 0.55:
        return "Partial ⚠️"
    else:
        return "Needs Improvement ❌"

In [23]:
# ── CELL 7: Per-Question Evaluator ──────────────────────────

def evaluate_question(question, user_answer, correct_answer, cluster_id, skills):
    scores = compute_hybrid_score(user_answer, correct_answer, question)  # ← add question
    grade  = get_grade(scores["hybrid_score"])
    return {
        "cluster_id":     cluster_id,
        "skills":         skills,
        "question":       question,
        "user_answer":    user_answer,
        "correct_answer": correct_answer,
        "grade":          grade,
        **scores
    }

In [24]:
# ── CELL 8: Per-Cluster Report Generator ────────────────────

def generate_cluster_feedback(cluster_results):
    clusters = defaultdict(list)
    for r in cluster_results:
        clusters[r["cluster_id"]].append(r)

    report = {}

    for cid, results in clusters.items():
        avg_score = np.mean([r["hybrid_score"] for r in results])
        skills    = results[0]["skills"]
        grade     = get_grade(avg_score)

        # Only flag weak metrics if score is actually low
        weak_metrics = []

        # Only check weak metrics for Partial or below
        if avg_score < 0.68:
            avg_bertscore = np.mean([r["bertscore"]   for r in results])
            avg_semantic  = np.mean([r["semantic_sim"] for r in results])
            avg_nli       = np.mean([r["nli_score"]    for r in results])
            avg_rouge     = np.mean([r["rouge_l"]      for r in results])

            if avg_rouge    < 0.40:
                weak_metrics.append("content coverage (key points missing)")
            if avg_nli      < 0.45:
                weak_metrics.append("logical correctness (answer may contradict expected)")
            if avg_semantic < 0.55:
                weak_metrics.append("semantic relevance (answer may be off-topic)")
            if avg_bertscore < 0.50:
                weak_metrics.append("terminology (key technical terms missing)")

        report[cid] = {
            "cluster_id":    cid,
            "skills":        skills,
            "avg_score":     round(avg_score, 3),
            "grade":         grade,
            "weak_metrics":  weak_metrics,
            "num_questions": len(results),
            "details":       results
        }

    return report


In [25]:
# # ── CELL 9: Full Evaluation Pipeline ────────────────────────

# def evaluate_interview(interview_data):
#     """
#     Main evaluation function.

#     interview_data: list of dicts, each containing:
#     {
#         "cluster_id":     int,
#         "skills":         list of str,
#         "question":       str,
#         "user_answer":    str,
#         "correct_answer": str
#     }

#     Returns: per-cluster feedback report
#     """
#     print("🔍 Evaluating answers...\n")
#     results = []

#     for i, item in enumerate(interview_data):
#         print(f"  Evaluating Q{i+1} (Cluster {item['cluster_id']})...")
#         result = evaluate_question(
#             question       = item["question"],
#             user_answer    = item["user_answer"],
#             correct_answer = item["correct_answer"],
#             cluster_id     = item["cluster_id"],
#             skills         = item["skills"]
#         )
#         results.append(result)

#     # Generate per-cluster feedback
#     report = generate_cluster_feedback(results)
#     return report


In [26]:
# # ── CELL 10: Print Report Function ──────────────────────────

# def print_report(report):
#     clusters_by_grade = {"excellent": [], "good": [], "partial": [], "missing": []}

#     for cid, data in report.items():
#         score = data["avg_score"]
#         if   score >= 0.78: clusters_by_grade["excellent"].append(data)
#         elif score >= 0.68: clusters_by_grade["good"].append(data)
#         elif score >= 0.55: clusters_by_grade["partial"].append(data)
#         else:               clusters_by_grade["missing"].append(data)

#     all_scores = [d["avg_score"] for d in report.values()]
#     overall    = np.mean(all_scores)

#     print("\n" + "="*60)
#     print("📋 INTERVIEW EVALUATION REPORT")
#     print("="*60)
#     print(f"\n🎯 Overall Score  : {overall:.2f} — {get_grade(overall)}")
#     print(f"📦 Total Clusters : {len(report)}")
#     print(f"   ✅ Excellent    : {len(clusters_by_grade['excellent'])}")
#     print(f"   👍 Good         : {len(clusters_by_grade['good'])}")
#     print(f"   ⚠️  Partial      : {len(clusters_by_grade['partial'])}")
#     print(f"   ❌ Needs Work   : {len(clusters_by_grade['missing'])}")

#     for label, emoji, key in [
#         ("SKILLS YOU EXCEL AT",        "✅", "excellent"),
#         ("SKILLS YOU ARE GOOD AT",     "👍", "good"),
#         ("SKILLS THAT NEED MORE WORK", "⚠️ ", "partial"),
#         ("SKILLS THAT ARE MISSING",    "❌", "missing"),
#     ]:
#         if clusters_by_grade[key]:
#             print(f"\n{'─'*60}")
#             print(f"{emoji} {label}:")
#             for d in clusters_by_grade[key]:
#                 print(f"\n  Cluster {d['cluster_id']} | Score: {d['avg_score']}")
#                 print(f"  Skills : {', '.join(d['skills'])}")
#                 print(f"  Grade  : {d['grade']}")
#                 if d["weak_metrics"]:
#                     print(f"  💡 To improve: {', '.join(d['weak_metrics'])}")

#     print(f"\n{'='*60}")


In [27]:
# # At the end of Cell 0 — runs automatically after interview
# report = evaluate_interview(interview_data)
# print_report(report)


In [28]:
# # ── DIAGNOSTIC CELL — run this before anything else ──────────

# # Pick one question from interview_data to debug
# test_item = interview_data[3]  # Q4 — IaC question (should be excellent)

# user_ans    = test_item["user_answer"]
# correct_ans = test_item["correct_answer"]
# question    = test_item["question"]

# print("="*60)
# print("DIAGNOSTIC: Individual Metric Scores")
# print("="*60)
# print(f"\nQuestion : {question[:100]}")
# print(f"\nUser     : {user_ans[:150]}")
# print(f"\nReference: {correct_ans[:150]}")

# # Test each metric individually
# print("\n--- Individual Scores ---")

# bs = compute_bertscore(user_ans, correct_ans)
# print(f"BERTScore F1     : {bs:.4f}")

# sem = compute_semantic_similarity(user_ans, correct_ans)
# print(f"Semantic Sim     : {sem:.4f}")

# nli = compute_nli_score(user_ans, correct_ans)
# print(f"NLI Score        : {nli:.4f}")

# rou = compute_rouge_l(user_ans, correct_ans)
# print(f"ROUGE-L          : {rou:.4f}")

# q_rel = compute_semantic_similarity(user_ans, question)
# print(f"Q Relevance      : {q_rel:.4f}")

# # Check if BERTScorer object exists
# print("\n--- Model Check ---")
# try:
#     print(f"bert_scorer type : {type(bert_scorer)}")
#     print(f"rescale_baseline : {bert_scorer.rescale_with_baseline}")
# except:
#     print("❌ bert_scorer NOT found — still using old function")

# # Show final hybrid
# scores = compute_hybrid_score(user_ans, correct_ans, question)
# print(f"\n--- Hybrid Score ---")
# for k, v in scores.items():
#     print(f"  {k}: {v}")

In [29]:
# # ── Test what an Excellent answer looks like ─────────────────
# test_item = interview_data[3]  # IaC question

# # Simulate user copying key phrases from reference
# reference  = test_item["correct_answer"]
# question   = test_item["question"]

# # Test 1: reference answer vs itself (should be ~1.0)
# scores_perfect = compute_hybrid_score(reference, reference, question)
# print(f"Perfect answer score  : {scores_perfect['hybrid_score']} — {get_grade(scores_perfect['hybrid_score'])}")

# # Test 2: actual user answer
# user_ans = test_item["user_answer"]
# scores_user = compute_hybrid_score(user_ans, reference, question)
# print(f"Your answer score     : {scores_user['hybrid_score']} — {get_grade(scores_user['hybrid_score'])}")

# # Test 3: completely wrong answer
# wrong = "I don't know anything about this topic at all."
# scores_wrong = compute_hybrid_score(wrong, reference, question)
# print(f"Wrong answer score    : {scores_wrong['hybrid_score']} — {get_grade(scores_wrong['hybrid_score'])}")




In [30]:
# # ── Calibration Test Cell ─────────────────────────────────────

# # Pick one question from your generated questions
# test_q    = all_questions[11]  # MySQL/PostgreSQL question
# question  = test_q["question"]
# reference = test_q["answer"]

# print("="*60)
# print("CALIBRATION TEST")
# print("="*60)
# print(f"Question  : {question}")
# print(f"Reference : {reference[:150]}...")
# print()

# test_cases = [
#     {
#         "type":     "✅ Correct Answer",
#         "expected": "Excellent",
#         "answer":   "Database administrators track changes in MySQL using the binary log which records all DDL and DML operations. In PostgreSQL the equivalent is the Write-Ahead Log (WAL). Schema migration tools like Liquibase and Flyway maintain a versioned history of structural changes. pgAudit provides detailed session-level auditing for compliance and monitoring."
#     },
#     {
#         "type":     "⚠️  Partial Answer",
#         "expected": "Partial",
#         "answer":   "You can use logs to track changes in the database and maybe some backup tools to keep a history of modifications."
#     },
#     {
#         "type":     "❌ Wrong Answer",
#         "expected": "Needs Improvement",
#         "answer":   "I am not sure about this topic. I think you can use some kind of file system."
#     }
# ]

# print(f"{'Type':<25} {'Expected':<20} {'Score':<10} {'Grade':<25} {'Pass'}")
# print("-"*90)

# all_passed = True
# for case in test_cases:
#     scores = compute_hybrid_score(case["answer"], reference, question)
#     grade  = get_grade(scores["hybrid_score"])
#     passed = "✅" if case["expected"].lower() in grade.lower() else "❌"
#     if "❌" in passed:
#         all_passed = False
#     print(f"{case['type']:<25} {case['expected']:<20} {scores['hybrid_score']:<10} {grade:<25} {passed}")

# print("-"*90)
# print(f"\n{'✅ All tests passed — system is calibrated correctly' if all_passed else '❌ Some tests failed — thresholds need adjustment'}")


# API

In [31]:
# ── CELL: Install FastAPI dependencies ───────────────────────
# ── Install ───────────────────────────────────────────────────
!pip install fastapi uvicorn pyngrok nest-asyncio -q

In [32]:
# ── Make all models available to API thread ───────────────────
import builtins

# Store everything in builtins so API thread can access them
builtins.bert_scorer_global  = bert_scorer
builtins.embedder_global     = embedder
builtins.nli_model_global    = nli_model
builtins.index_global        = index
builtins.data_global         = data
builtins.model_sim_global    = model_sim
builtins.gen_model_global    = gen_model
builtins.gen_tokenizer_global = gen_tokenizer
builtins.gemini_client_global = client

print("✅ All models stored in builtins")

✅ All models stored in builtins


In [33]:
# ── CELL: Single Route API ────────────────────────────────────
import builtins
import nest_asyncio
import uvicorn
import threading
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
from pydantic import BaseModel
from typing import List
# Add this as first line in API cell
from pyngrok import ngrok
ngrok.kill()
import time
time.sleep(2)
nest_asyncio.apply()
index       = builtins.index_global
data        = builtins.data_global
model_sim   = builtins.model_sim_global

nest_asyncio.apply()

app = FastAPI(title="Interview Preparation Module")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Request Model ─────────────────────────────────────────────
class InterviewRequest(BaseModel):
    job_description: str
    answers: List[dict] = []
    # answers is empty on first call (generate questions)
    # answers is filled on second call (evaluate)
    # Each answer dict:
    # { "question": "...", "user_answer": "...",
    #   "correct_answer": "...", "skills": [...], "cluster_id": 0 }

# ── Add this helper function in the API cell ─────────────────
import tempfile
import os

def extract_skills_from_text(job_description_text: str):
    """
    Wrapper that converts raw text to a temp file
    so parse_jd_tech_skills_with_llm can read it
    """
    # Write text to a temporary file
    with tempfile.NamedTemporaryFile(
        mode='w',
        suffix='.txt',
        delete=False,
        encoding='utf-8'
    ) as tmp:
        tmp.write(job_description_text)
        tmp_path = tmp.name

    try:
        # Call original function with file path
        result = parse_jd_tech_skills_with_llm(tmp_path)
        return result.get("jd_tech_skills", [])
    finally:
        # Clean up temp file
        os.unlink(tmp_path)

# ── Single Route ──────────────────────────────────────────────
@app.post("/interview")
def interview(request: InterviewRequest):
    try:
        if not request.answers:

            # Step 1: Extract skills from raw text
            skills = extract_skills_from_text(request.job_description)

            if not skills:
                raise HTTPException(
                    status_code=400,
                    detail="No skills could be extracted from the job description"
                )

            # Step 2: Cluster skills
            clustering_result = cluster_skills_kmeans(
                skills, model_sim, k=None, k_min=2, k_max=10
            )

            # Step 3: Generate questions
            questions = generate_rag_questions_for_clusters(
                clustering_result, index, data, embedder,
                questions_per_cluster=2
            )

            return {
                "status":    "questions_ready",
                "skills":    skills,
                "questions": [
                    {
                        "cluster_id": q["cluster_id"],
                        "skills":     q["skills"],
                        "question":   q["question"],
                        "answer":     q["answer"],
                    }
                    for q in questions
                ]
            }

        else:
            results = []
            for item in request.answers:
                scores = compute_hybrid_score(
                    item["user_answer"],
                    item["correct_answer"],
                    item["question"]
                )
                grade = get_grade(scores["hybrid_score"])
                results.append({
                    "cluster_id":     item["cluster_id"],
                    "skills":         item["skills"],
                    "question":       item["question"],
                    "user_answer":    item["user_answer"],
                    "correct_answer": item["correct_answer"],
                    "grade":          grade,
                    **scores
                })

            report        = generate_cluster_feedback(results)
            all_scores    = [d["avg_score"] for d in report.values()]
            overall       = round(float(np.mean(all_scores)), 3)
            overall_grade = get_grade(overall)

            clusters = {}
            for cid, d in report.items():
                clusters[str(cid)] = {
                    "cluster_id":   d["cluster_id"],
                    "skills":       d["skills"],
                    "avg_score":    d["avg_score"],
                    "grade":        d["grade"],
                    "weak_metrics": d["weak_metrics"],
                }

            return {
                "status":        "report_ready",
                "overall_score": overall,
                "overall_grade": overall_grade,
                "clusters":      clusters
            }

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── Health Check ──────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "running", "module": "Interview Preparation"}


@app.get("/test_eval")
def test_eval():
    try:
        score = compute_hybrid_score(
            "Docker containers package applications into isolated environments using a Dockerfile.",
            "Docker is a platform for containerizing applications using Dockerfiles and images.",
            "How does Docker work?"
        )
        return {"status": "ok", "score": score}
    except Exception as e:
        return {"status": "error", "detail": str(e)}
"""

Then test it in Postman:
```
GET https://demetrice-unpitiful-blunderingly.ngrok-free.dev/test_eval
"""

# ── Run ───────────────────────────────────────────────────────
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

ngrok.set_auth_token("3AyrPhfkYT2Hc4RcKOszX10VXLJ_G3fuf8WE9JVc4ZBhYY6v")  # ← replace this
public_url = ngrok.connect(8000)

print("="*60)
print(f"✅ API is live at: {public_url}")
print(f"📖 Docs: {public_url}/docs")
print("="*60)
print(f"\nShare with backend team:")
print(f"  Route: POST {public_url}/interview")

thread = threading.Thread(target=run)
thread.start()
"""

---

## What to Tell Your Colleague

Give her exactly this:
```
Route: POST /interview

── Call 1: Generate Questions ──
Request:
{
  "job_description": "We are looking for a DevOps engineer..."
}

Response:
{
  "status": "questions_ready",
  "skills": ["docker", "aws", "ci/cd", ...],
  "questions": [
    {
      "cluster_id": 0,
      "skills": ["docker", "ecs"],
      "question": "How do you create a Docker image?",
      "answer": "ideal answer for evaluation"
    },
    ...
  ]
}

── Call 2: Evaluate Answers ──
Request:
{
  "job_description": "...",
  "answers": [
    {
      "cluster_id": 0,
      "skills": ["docker", "ecs"],
      "question": "How do you create a Docker image?",
      "user_answer": "user typed or spoke answer here",
      "correct_answer": "ideal answer from call 1"
    }
  ]
}

Response:
{
  "status": "report_ready",
  "overall_score": 0.72,
  "overall_grade": "Good 👍",
  "clusters": {
    "0": {
      "skills": ["docker", "ecs"],
      "avg_score": 0.75,
      "grade": "Excellent ✅",
      "weak_metrics": []
    }
  }
}
"""

✅ API is live at: NgrokTunnel: "https://demetrice-unpitiful-blunderingly.ngrok-free.dev" -> "http://localhost:8000"
📖 Docs: NgrokTunnel: "https://demetrice-unpitiful-blunderingly.ngrok-free.dev" -> "http://localhost:8000"/docs

Share with backend team:
  Route: POST NgrokTunnel: "https://demetrice-unpitiful-blunderingly.ngrok-free.dev" -> "http://localhost:8000"/interview


'\n\n---\n\n## What to Tell Your Colleague\n\nGive her exactly this:\n```\nRoute: POST /interview\n\n── Call 1: Generate Questions ──\nRequest:\n{\n  "job_description": "We are looking for a DevOps engineer..."\n}\n\nResponse:\n{\n  "status": "questions_ready",\n  "skills": ["docker", "aws", "ci/cd", ...],\n  "questions": [\n    {\n      "cluster_id": 0,\n      "skills": ["docker", "ecs"],\n      "question": "How do you create a Docker image?",\n      "answer": "ideal answer for evaluation"\n    },\n    ...\n  ]\n}\n\n── Call 2: Evaluate Answers ──\nRequest:\n{\n  "job_description": "...",\n  "answers": [\n    {\n      "cluster_id": 0,\n      "skills": ["docker", "ecs"],\n      "question": "How do you create a Docker image?",\n      "user_answer": "user typed or spoke answer here",\n      "correct_answer": "ideal answer from call 1"\n    }\n  ]\n}\n\nResponse:\n{\n  "status": "report_ready",\n  "overall_score": 0.72,\n  "overall_grade": "Good 👍",\n  "clusters": {\n    "0": {\n      "sk